In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)


sns.set(style="whitegrid")

In [ ]:
DATA_PATH = "../outputs"

# Archivos
df_temp_path = os.path.join(DATA_PATH, "features_temporal_clean.parquet")
df_spa_path  = os.path.join(DATA_PATH, "features_spatial_clean.parquet")
df_freq_path = os.path.join(DATA_PATH, "features_frequency_clean.parquet")

df_temp = pd.read_parquet(df_temp_path)
df_spa   = pd.read_parquet(df_spa_path)
df_freq  = pd.read_parquet(df_freq_path)

# Verificar shapes
print(f"Temporal: {df_temp.shape}, Espacial: {df_spa.shape}, Frecuencia: {df_freq.shape}")


Check: Todos los datasets tienen el mismo número de epochs, aunque distinto número de columnas (es correcto)

Observamos uno de los datasets

In [ ]:
df_freq

## Numero de sujetos

In [ ]:
print("Se han analizado",len(df_freq["subject"].unique()),"sujetos.")

In [ ]:
def analyze_features_subject(df):
    """
    Análisis a nivel de sujeto para un único dataframe.

    - Columna 1: número de epochs por sujeto
    - Columna 2: distribución de clases por sujeto
    """

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(14, 4),
        squeeze=False
    )

    # ===============================
    # 1. Número de epochs por sujeto
    # ===============================
    epochs_per_subject = df.groupby("subject").size()

    axes[0, 0].bar(
        epochs_per_subject.index.astype(str),
        epochs_per_subject.values
    )
    axes[0, 0].set_xlabel("Sujeto")
    axes[0, 0].set_ylabel("Número de epochs")
    axes[0, 0].set_title("Distribución de epochs por sujeto")

    # ==================================
    # 2. Distribución de clases por sujeto
    # ==================================
    class_per_subject = (
        df.groupby(["subject", "label"])
          .size()
          .unstack(fill_value=0)
    )

    class_per_subject.plot(
        kind="bar",
        stacked=True,
        ax=axes[0, 1],
        legend=True
    )

    axes[0, 1].set_xlabel("Sujeto")
    axes[0, 1].set_ylabel("Número de epochs")
    axes[0, 1].set_title("Distribución de clases por sujeto")

    plt.tight_layout()
    plt.show()

Distribución de clases

In [ ]:
df_freq.groupby("label")["epoch"].count()

Distribución de epochs por archivo

In [ ]:
df_freq.groupby("subject")["epoch"].count()

Distribución de epochs por cada sujeto y clase

In [ ]:
df_freq.groupby(["subject", "label"])["epoch"].count()


In [ ]:
df_freq.groupby("subject")["epoch"].count().min(), df_freq.groupby("subject")["epoch"].count().max()

Cada archivo edf registró entre 104 y 840 epochs.

In [ ]:
analyze_features_subject(df_freq)

Dependencia intra-sujeto

In [ ]:
def analyze_subject_dependence(df):
    """
    Diagnóstico de dependencia intra-sujeto y riesgo de data leakage
    """

    print("\n=== Diagnóstico de estructura por sujeto ===")

    # 1. Distribución de epochs
    epochs_per_subject = df.groupby("subject").size()

    imbalance_ratio = epochs_per_subject.max() / epochs_per_subject.min()
    top_20_pct = epochs_per_subject.sort_values(ascending=False).head(
        max(1, int(0.2 * len(epochs_per_subject)))
    ).sum() / len(df)

    print(f"\nDistribución de epochs:")
    print(f"Ratio max/min: {imbalance_ratio:.2f}")
    print(f"% epochs top 20% sujetos: {top_20_pct:.2%}")

    # 2. Dependencia intra-sujeto
    numeric_features = (
        df.select_dtypes(include=np.number)
          .columns
          .drop(["label", "subject"], errors="ignore")
    )

    intra_var = (
        df.groupby("subject")[numeric_features]
          .var()
          .mean()
    )
    global_var = df[numeric_features].var()

    ratio_mean = (intra_var / global_var).mean()

    print(f"\nDependencia intra-sujeto:")
    print(f"Ratio medio var intra / global: {ratio_mean:.2f}")

    # 3. Riesgo de fuga

    train_df, val_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df["label"]
    )

    overlap = set(train_df["subject"]).intersection(val_df["subject"])
    leakage_ratio = len(overlap) / df["subject"].nunique()

    print(f"\nRiesgo de fuga de información:")
    print(f"Sujetos compartidos train/val: {len(overlap)}")
    print(f"Proporción de fuga: {leakage_ratio:.2%}")


In [ ]:
analyze_subject_dependence(df_freq)

In [ ]:
analyze_subject_dependence(df_spa)

In [ ]:
analyze_subject_dependence(df_temp)

## Información general

Todas las variables predictoras de los 3 dataset son variables numéricas. La variable objetivo es una variable binaria.

In [ ]:
def outliers_detect(varaux):
    """
    Identifica valores atípicos en una serie numérica y los marca como NaN
    (solo en la serie devuelta, no modifica el dataframe original).

    Criterio:
    1) Según la forma de la distribución:
       - Distribución aproximadamente simétrica (|skew| < 1):
         se usa z-score con umbral |z| > 3
       - Distribución asimétrica:
         se usa desviación absoluta de la mediana (MAD) con umbral > 8

    2) Según rango intercuartílico ampliado:
       - Q1 = percentil 25, Q3 = percentil 75
       - H = 3 · (Q3 − Q1)
       - Valores fuera de [Q1 − H, Q3 + H]

    Un valor se considera atípico solo si cumple ambos criterios
    simultáneamente.

    Parámetros
    ----------
    varaux : pandas.Series
        Serie numérica a analizar.

    Retorna
    -------
    list:
        [serie_con_atipicos_como_NaN, numero_de_atipicos]
    """

    import numpy as np
    import statsmodels.api as sm

    varaux = varaux.copy()

    if abs(varaux.skew()) < 1:
        criterio1 = abs((varaux - varaux.mean()) / varaux.std()) > 3
    else:
        mad = sm.robust.mad(varaux, axis=0)
        criterio1 = abs((varaux - varaux.median()) / mad) > 8

    qnt = varaux.quantile([0.25, 0.75]).dropna()
    Q1 = qnt.iloc[0]
    Q3 = qnt.iloc[1]
    H = 3 * (Q3 - Q1)

    criterio2 = (varaux < (Q1 - H)) | (varaux > (Q3 + H))

    var = varaux.copy()
    mask_atipicos = criterio1 & criterio2
    var[mask_atipicos] = np.nan

    return [var, mask_atipicos.sum()]


def top_feature_correlations(df_predictors, feature_type, top_n=10, corr_threshold=0.7, plot=False):
    """
    Muestra los pares de variables predictoras con mayor correlación absoluta,
    indicando si es positiva o negativa, y opcionalmente plotea las correlaciones
    por encima de un umbral.

    Parameters
    ----------
    df_predictors : pd.DataFrame
        DataFrame con solo columnas predictoras (numéricas)
    top_n : int
        Número de pares top a mostrar
    corr_threshold : float
        Solo se muestran/plotearán pares con |correlación| >= corr_threshold
    plot : bool
        Si True, se genera un heatmap de las correlaciones filtradas

    Returns
    -------
    pd.DataFrame con columnas: ['Feature1', 'Feature2', 'Correlation']
    """
    import pandas as pd
    import numpy as np
    import seaborn as sns
    import matplotlib.pyplot as plt

    # Solo columnas numéricas
    df_num = df_predictors.select_dtypes(include=np.number)

    # Matriz de correlación
    corr_matrix = df_num.corr()

    # Tomar solo la parte superior de la matriz sin la diagonal
    corr_matrix_upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Aplanar
    corr_pairs = corr_matrix_upper.stack().reset_index()
    corr_pairs.columns = ['Feature1', 'Feature2', 'Correlation']

    # Filtrar por umbral
    corr_pairs = corr_pairs[ corr_pairs['Correlation'].abs() >= corr_threshold ]

    # Ordenar por valor absoluto de correlación
    corr_pairs['AbsCorr'] = corr_pairs['Correlation'].abs()
    corr_pairs_sorted = corr_pairs.sort_values('AbsCorr', ascending=False)

    # Seleccionar top_n
    top_corr = corr_pairs_sorted.head(top_n).copy()
    top_corr.drop(columns='AbsCorr', inplace=True)

    # Imprimir
    print(f"\nTop {len(top_corr)} pares de features con |correlación| >= {corr_threshold} para FEATURES {feature_type.upper()}:")
    for idx, row in top_corr.iterrows():
        signo = "Positiva" if row['Correlation'] > 0 else "Negativa"
        print(f"{row['Feature1']} ↔ {row['Feature2']}: {row['Correlation']:.3f} ({signo})")

    # Plot opcional
    if plot and not top_corr.empty:
        plt.figure(figsize=(8, len(top_corr)*0.4+3))
        sns.heatmap(
            corr_matrix.loc[top_corr['Feature1'], top_corr['Feature2']],
            annot=True, fmt=".2f", cmap='coolwarm', cbar=True
        )
        plt.title(f"Correlaciones > {corr_threshold}")
        plt.tight_layout()
        plt.show()

    return top_corr

def plot_cramers_v(df_predictors, target, feature_type , n_bins=10, top_n=10):
    """
    Calcula y grafica el V de Cramér entre cada feature y el target binario (0/1)
    mostrando solo las top_n features más asociadas.

    Parameters
    ----------
    df_predictors : pd.DataFrame
        DataFrame con SOLO columnas predictoras
    target : pd.Series
        Variable objetivo binaria (0/1)
    n_bins : int
        Número de bins para discretizar variables continuas
    top_n : int
        Número de features top a mostrar en el gráfico
    """
    from scipy.stats import chi2_contingency
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

    def cramers_v(x, y):
        contingency = pd.crosstab(x, y)
        if contingency.shape[0] < 2 or contingency.shape[1] < 2:
            return 0.0
        chi2 = chi2_contingency(contingency)[0]
        n = contingency.sum().sum()
        r, k = contingency.shape
        return np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

    cramer_values = {}
    for col in df_predictors.columns:
        x = df_predictors[col]

        if not np.issubdtype(x.dtype, np.number):
            continue

        try:
            x_binned = pd.qcut(x, q=n_bins, duplicates='drop')
            cramer_values[col] = cramers_v(x_binned, target)
        except ValueError:
            cramer_values[col] = 0.0

    cramer_df = (
        pd.DataFrame.from_dict(cramer_values, orient='index', columns=['cramers_v'])
        .sort_values('cramers_v', ascending=False)  # descendente para top
    )

    # Seleccionar solo las top_n features
    cramer_df_top = cramer_df.head(top_n)[::-1]  # invertir para barras horizontales

    # Plot
    plt.figure(figsize=(8, max(4, len(cramer_df_top) * 0.5)))
    plt.barh(cramer_df_top.index, cramer_df_top['cramers_v'], color='skyblue')
    plt.xlabel("V de Cramér")
    plt.title(f"Top {len(cramer_df_top)} features vs Target ({feature_type})")
    plt.tight_layout()
    plt.show()

    return cramer_df_top

def analyze_features(df, feature_type):

    print(f"\n=== Análisis de features: {feature_type} ===")

    feature_cols = [c for c in df.columns if c not in ['subject', 'epoch', 'label']]

    # =====================================================
    # Análisis de NaNs existentes
    # =====================================================
    n_missing_total = df[feature_cols].isna().sum().sum()
    n_missing_per_feature = df[feature_cols].isna().sum()
    n_features_with_nans = (n_missing_per_feature > 0).sum()

    print(f"\nTotal de valores faltantes (solo features): {n_missing_total}")
    print(f"Features con al menos un NaN: {n_features_with_nans}")

    if n_features_with_nans > 0:
        print("\nNaNs por feature (features con NaNs):")
        print(n_missing_per_feature[n_missing_per_feature > 0].sort_values(ascending=False))

    # =====================================================
    # Análisis de valores atípicos (solo diagnóstico)
    # =====================================================
    print("\n=== Análisis de valores atípicos (diagnóstico) ===")

    resultados_atipicos = {}
    atipicos_por_clase = {label: 0 for label in df['label'].unique()}

    for col in feature_cols:
        if pd.api.types.is_numeric_dtype(df[col]):
            _, n_atipicos = outliers_detect(df[col])
            prop_atipicos = n_atipicos / len(df)
            resultados_atipicos[col] = prop_atipicos

            # Distribución de atípicos por clase (sin modificar df)
            serie_atipicos, _ = outliers_detect(df[col])
            idx_atipicos = serie_atipicos.isna()

            for label in atipicos_por_clase:
                atipicos_por_clase[label] += idx_atipicos[df['label'] == label].sum()

    resultados_atipicos_filtrados = {
        var: prop for var, prop in resultados_atipicos.items() if prop > 0.01
    }

    print(f"Features numéricas analizadas: {len(resultados_atipicos)}")
    print(f"Features con >1% de atípicos: {len(resultados_atipicos_filtrados)}")

    if len(resultados_atipicos_filtrados) > 0:
        print("\nProporción de atípicos por feature (>1%):")
        for var, prop in sorted(resultados_atipicos_filtrados.items(),
                                key=lambda x: x[1], reverse=True):
            print(f"{var}: {prop:.2%}")

    print("\nResumen global de atípicos:")
    print(f"Proporción media: {np.mean(list(resultados_atipicos.values())):.2%}")
    print(f"Proporción máxima: {np.max(list(resultados_atipicos.values())):.2%}")

    # =====================================================
    # Atípicos por clase
    # =====================================================
    print("\n=== Atípicos acumulados por clase ===")

    total_atipicos = sum(atipicos_por_clase.values())

    for label, count in sorted(atipicos_por_clase.items(),
                               key=lambda x: x[1], reverse=True):
        prop = count / total_atipicos if total_atipicos > 0 else 0
        print(f"Clase {label}: {count} atípicos ({prop:.2%})")

    # =====================================================
    # Información general
    # =====================================================
    print("\nInformación general del dataframe:")
    df.info()

    print("\nEstadísticas descriptivas:")
    display(df[feature_cols].describe())

    # =====================================================
    # Visualizaciones
    # =====================================================
    plt.figure(figsize=(5, 3))
    sns.countplot(x='label', data=df)
    plt.title(f"Distribución de clases - {feature_type}")
    plt.tight_layout()
    plt.show()

    n_plot_features = min(20, len(feature_cols))

    plt.figure(figsize=(12, 5))
    sns.boxplot(data=df[feature_cols[:n_plot_features]])
    plt.title(f"Boxplot primeras {n_plot_features} features - {feature_type}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    top_feature_correlations(df.drop(columns=['label', 'epoch', 'subject']), feature_type=feature_type)
    plot_cramers_v(df.drop(columns=['label', 'epoch', 'subject']), df['label'], feature_type=feature_type)

    print("\n=== Resumen final ===")
    print(f"Total filas: {len(df)}")
    print(f"Total features: {len(feature_cols)}")

## Features de Frecuencia

In [ ]:
analyze_features(df_freq, "Frecuencia")

Para las features de frecuencia:

* Se observaron 1337216 valores faltantes.
* Observamos un elevado desablanceo de clases, con predominancia de la clase no objetivo.
* Observamos un elevado número de observaciones atípicas en un gran número de variables. Ninguna alcanza el 3% de atípicos.
* Observamos alta correlacion positiva entre las ondas delta de diferentes canales, superior aal 98%. Especialmente la de los canales 10, 11, 12, 13, 14 entre sí y la del canal 5 con la de los canales 10, 11, 12.  Por tanto, considerar todas estas variables implica aportar
dependencias y multicolinealidad al modelo.
* Observamos que las ondas alfa y beta del canal 5 son las que mas influyen en la variable objetivo. Le siguen los canales 15, 4 ,8 con ondas delta, alpha, delta, respectivamente.

## Features Espaciales

In [ ]:
analyze_features(df_spa, "Espacial")

Para las features espaciales:

* No se observaron valores faltantes.
* Observamos un elevado desbalanceo de clases, con predominancia de la clase no objetivo.
* Observamos un elevado número de observaciones atípicas en todas las variables. Algunas alcanzado el 7% de atípicos.
* Observamos alta correlacion entre las variables, superior al 80%, pues están relacionadas matemáticamente. Por tanto, considerar todas estas variables implica aportar
dependencias y multicolinealidad al modelo.
* Observamos que el rango, la std y el mínimo son las variables con mayor influencia, en ese orden.

## Features Temporales

In [ ]:
analyze_features(df_temp, "Temporal")

Para las features temporales:

* Observamos que la entropía tiene un valor significativamente superior a 0 mientras que el resto de variables son del orden e-n
* No se observaron valores faltantes.
* Observamos un elevado desbalanceo de clases, con predominancia de la clase no objetivo.
* Observamos un elevado número de observaciones atípicas un elevado número de variables. Algunas alcanzando el 10% valores atípicos.
* Observamos alta correlacion entre la energía de los canales 5, 10, 11, 12, 13, 14, superior al 97%.  Por tanto, considerar todas estas variables implica aportar
dependencias y multicolinealidad al modelo.
* La media y la mediana del canal 1 son las variables con mayor impacto sobre la variable objetivo, seguidas de la energía, rms, std del canal 5

## Analisis por sujeto

In [ ]:
def analyze_subject_dependence(df):
    """
    Diagnóstico de dependencia intra-sujeto y riesgo de data leakage
    """

    print("\n=== Diagnóstico de estructura por sujeto ===")

    # 1. Distribución de epochs
    epochs_per_subject = df.groupby("subject").size()

    imbalance_ratio = epochs_per_subject.max() / epochs_per_subject.min()
    top_20_pct = epochs_per_subject.sort_values(ascending=False).head(
        max(1, int(0.2 * len(epochs_per_subject)))
    ).sum() / len(df)

    print(f"\nDistribución de epochs:")
    print(f"Ratio max/min: {imbalance_ratio:.2f}")
    print(f"% epochs top 20% sujetos: {top_20_pct:.2%}")

    # 2. Dependencia intra-sujeto
    numeric_features = (
        df.select_dtypes(include=np.number)
          .columns
          .drop(["label", "subject"], errors="ignore")
    )

    intra_var = (
        df.groupby("subject")[numeric_features]
          .var()
          .mean()
    )
    global_var = df[numeric_features].var()

    ratio_mean = (intra_var / global_var).mean()

    print(f"\nDependencia intra-sujeto:")
    print(f"Ratio medio var intra / global: {ratio_mean:.2f}")

    # 3. Riesgo de fuga
    from sklearn.model_selection import train_test_split

    train_df, val_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df["label"]
    )

    overlap = set(train_df["subject"]).intersection(val_df["subject"])
    leakage_ratio = len(overlap) / df["subject"].nunique()

    print(f"\nRiesgo de fuga de información:")
    print(f"Sujetos compartidos train/val: {len(overlap)}")
    print(f"Proporción de fuga: {leakage_ratio:.2%}")

    if leakage_ratio > 0:
        print("⚠️ Recomendado: GroupKFold o Leave-One-Subject-Out")
